# Test 1: does the model predict, or remember?

The plan was committed in `24ba7c3`, and Llama 2's part in `b3baef3`, both before any of these answers existed: see [PLAN.md](PLAN.md).

- **Model:** `llama-2-7b-chat`, 4-bit, run on this laptop with llama.cpp. Its pretraining data ends in Sep 2022, and some fine-tuning data in Jul 2023.
- **Three ways of asking:** *named* (index name, dates, 20 days of prices), *hidden* (the same prices with name and dates removed), *date only* (no prices).
- **Three periods:** *before* (Jan 2017 – Aug 2022), *in between* (Sep 2022 – Jul 2023, every day) and *after* (Aug 2023 – Oct 2026), 300 random days per market in *before* and *after*.
- **Main score:** AUC. 0.50 is a coin flip.

This notebook only reads the saved answers. `run_accuracy.py` collects them.

In [1]:
import sys
sys.path.insert(0, "..")  # the main project's code is one folder up

import json

import pandas as pd
from sklearn.metrics import roc_auc_score

from data import MARKETS, clean, load_market
from evaluation import bootstrap_range, holm, majority_direction, mcnemar
from llm_harness.answers import last_word, up_probability
from llm_harness.client import SAVED
from make_fake_data import fake_prices

RUN = "accuracy_llama2"
PERIODS = ["before", "in between", "after"]
MODES = ["hidden", "named", "date only"]

prices = {market: load_market(market, include_holdout=True) for market in MARKETS}
prices["fake"] = clean(fake_prices(seed=99), "fake")

## The answers

One row per question: what the model said, what happened, and what the fair bar would have said.

In [2]:
def fair_calls(df):
    """For each year, the direction that was more common in all earlier days."""
    up = df["ret"].dropna() > 0
    return {year: majority_direction(up[up.index.year < year]) for year in range(2012, 2027)}

fair = {market: fair_calls(df) for market, df in prices.items()}

rows = []
for line in (SAVED / f"{RUN}.jsonl").read_text(encoding="utf-8").splitlines():
    record = json.loads(line)
    day, market, reply = pd.Timestamp(record["day"]), record["market"], record["reply"]
    thinking = record["mode"].endswith("thinking")
    rows.append({
        "day": day, "market": market, "period": record["period"], "mode": record["mode"],
        "word": last_word(reply) if thinking else (reply["choices"][0]["message"]["content"] or "").strip().lower().rstrip(".!"),  # "Up." counts as "up"
        "p_up": None if thinking else up_probability(reply),
        "up": bool(prices[market].loc[day, "ret"] > 0),
        "fair_call": fair[market][day.year],
    })
answers = pd.DataFrame(rows).set_index("day").sort_index()

unclear = ~answers["word"].isin(["up", "down"])
print(f"{len(answers)} answers, {unclear.sum()} of them neither up nor down (left out)")
answers = answers[~unclear]
answers["call_up"] = answers["word"] == "up"
real = answers[answers["market"] != "fake"]

15126 answers, 0 of them neither up nor down (left out)


## Overview

Every way of asking in every period, six markets together. **p vs fair** is McNemar's paired test of accuracy against the fair bar.

In [3]:
def overview(group):
    has_probabilities = group["p_up"].notna().all()
    return pd.Series({
        "questions": len(group),
        "said up %": 100 * group["call_up"].mean(),
        "days up %": 100 * group["up"].mean(),
        "accuracy %": 100 * (group["call_up"] == group["up"]).mean(),
        "fair bar %": 100 * (group["fair_call"] == group["up"]).mean(),
        "p vs fair": mcnemar(group["up"], group["call_up"], group["fair_call"])[2],
        "AUC": roc_auc_score(group["up"], group["p_up"].astype(float)) if has_probabilities else float("nan"),
    })

table = real.groupby(["period", "mode"])[real.columns].apply(overview)
table.loc[[(period, mode) for period in PERIODS for mode in MODES]].round(3)

questions  said up %  days up %  accuracy %  fair bar %  \
period     mode                                                                 
before     hidden        1800.0    100.000     51.444      51.444      51.444   
           named         1800.0    100.000     51.444      51.444      51.444   
           date only     1800.0     63.000     51.444      51.667      51.444   
in between hidden        1342.0    100.000     50.075      50.075      50.075   
           named         1342.0    100.000     50.075      50.075      50.075   
           date only     1342.0     59.613     50.075      49.627      50.075   
after      hidden        1800.0    100.000     52.611      52.611      52.611   
           named         1800.0    100.000     52.611      52.611      52.611   
           date only     1800.0     59.056     52.611      51.444      52.611   

                      p vs fair    AUC  
period     mode                         
before     hidden         1.000  0.478  
           named          1.000  0.502  
           date only      0.907  0.525  
in between hidden         1.000  0.507  
           named          1.000  0.492  
           date only      0.830  0.503  
after      hidden         1.000  0.484  
           named          1.000  0.510  
           date only      0.461  0.497

## The three planned comparisons

AUC with its 95% range (whole weeks resampled 2,000 times).

1. **Hidden:** can it predict from prices alone?
2. **Named minus hidden,** on the same days: do the name and dates add anything?
3. **Date only:** pure memory.

Memory would show as comparison 2 or 3 above its coin-flip value *before* the cutoff and not *after*.

In [4]:
def auc(frame, column="p_up"):
    return roc_auc_score(frame["up"], frame[column].astype(float))

def with_range(daily, statistic):
    """The number itself and its 95% range."""
    low, high = bootstrap_range(daily, statistic)
    return {"value": statistic(daily), "low": low, "high": high}

def one(mode, period):
    return real[(real["mode"] == mode) & (real["period"] == period)]

def side_by_side(first, second, period):
    """Two ways of asking, matched on the same market and day."""
    left = one(first, period).set_index("market", append=True)
    right = one(second, period).set_index("market", append=True)[["p_up", "call_up"]].add_suffix("_other")
    return left.join(right, how="inner").reset_index("market")

comparisons = {}
for period in PERIODS:
    comparisons["1. hidden (coin flip = 0.50)", period] = with_range(one("hidden", period), auc)
    comparisons["2. named minus hidden (no effect = 0)", period] = with_range(
        side_by_side("named", "hidden", period), lambda both: auc(both) - auc(both, "p_up_other"))
    comparisons["3. date only (coin flip = 0.50)", period] = with_range(one("date only", period), auc)
comparisons = pd.DataFrame(comparisons).T
comparisons.round(3)

,,value,low,high
1. hidden (coin flip = 0.50),before,0.478,0.452,0.503
2. named minus hidden (no effect = 0),before,0.024,-0.001,0.051
3. date only (coin flip = 0.50),before,0.525,0.497,0.552
1. hidden (coin flip = 0.50),in between,0.507,0.473,0.539
2. named minus hidden (no effect = 0),in between,-0.015,-0.051,0.022
3. date only (coin flip = 0.50),in between,0.503,0.475,0.533
1. hidden (coin flip = 0.50),after,0.484,0.458,0.511
2. named minus hidden (no effect = 0),after,0.026,-0.000,0.053
3. date only (coin flip = 0.50),after,0.497,0.469,0.526


The same three comparisons on accuracy, with paired tests. **After adjusting** uses Holm's method within each comparison.

In [5]:
tests = {}
for period in PERIODS:
    hidden, date_only, both = one("hidden", period), one("date only", period), side_by_side("named", "hidden", period)
    tests["1. hidden vs fair bar", period] = mcnemar(hidden["up"], hidden["call_up"], hidden["fair_call"])
    tests["2. named vs hidden", period] = mcnemar(both["up"], both["call_up"], both["call_up_other"])
    tests["3. date only vs fair bar", period] = mcnemar(date_only["up"], date_only["call_up"], date_only["fair_call"])
tests = pd.DataFrame(tests, index=["days only the first got right", "days only the second got right", "p"]).T
tests["p after adjusting"] = tests["p"].groupby(level=0, group_keys=False).apply(holm)
tests.round(3)

,,days only the first got right,days only the second got right,p,p after adjusting
1. hidden vs fair bar,before,0.0,0.0,1.000,1.0
2. named vs hidden,before,0.0,0.0,1.000,1.0
3. date only vs fair bar,before,335.0,331.0,0.907,1.0
1. hidden vs fair bar,in between,0.0,0.0,1.000,1.0
2. named vs hidden,in between,0.0,0.0,1.000,1.0
3. date only vs fair bar,in between,268.0,274.0,0.830,1.0
1. hidden vs fair bar,after,0.0,0.0,1.000,1.0
2. named vs hidden,after,0.0,0.0,1.000,1.0
3. date only vs fair bar,after,358.0,379.0,0.461,1.0


## Check on the harness: fake data

Made-up random prices, asked the hidden way. Nothing can be predicted, so AUC should be about 0.50.

In [6]:
fake = answers[answers["market"] == "fake"]
display(overview(fake).round(3).to_frame("fake data").T)
pd.Series(with_range(fake, auc), name="AUC on fake data").round(3).to_frame().T

,questions,said up %,days up %,accuracy %,fair bar %,p vs fair,AUC
fake data,300.0,100.0,50.667,50.667,49.333,0.708,0.422


,value,low,high
AUC on fake data,0.422,0.359,0.483


## A closer look by year (not in the plan)

This table wasn't in the plan, but it was written before the answers were seen. The plan expected memory to show in famous stretches such as March 2020, so this shows AUC by year for the two ways of asking that could use memory.

In [7]:
by_year = {}
for mode in ["date only", "named", "hidden"]:
    group = real[real["mode"] == mode]
    by_year[mode] = {year: auc(in_year) for year, in_year in group.groupby(group.index.year) if in_year["up"].nunique() == 2}
by_year = pd.DataFrame(by_year)
by_year["questions"] = real[real["mode"] == "hidden"].groupby(lambda day: day.year).size()
by_year.round(3)

,date only,named,hidden,questions
2017,0.517,0.505,0.524,305
2018,0.524,0.503,0.500,307
2019,0.569,0.511,0.446,311
2020,0.536,0.553,0.499,323
2021,0.490,0.476,0.460,343
2022,0.499,0.495,0.463,710
2023,0.496,0.489,0.517,1065
2024,0.535,0.504,0.435,560
2025,0.498,0.505,0.494,567
2026,0.465,0.508,0.519,451


## What this shows

Like Gemma: no skill from prices, and no clear sign of memory. Two numbers sit close to their limits and are reported as they came out.

- **With prices, Llama 2 says "Up" to every question:** 100% in every period. Its accuracy is exactly the fair bar's.
- **Prices alone: no skill.** Hidden AUC is 0.478 before (range 0.452 to 0.503), 0.507 in between and 0.484 after.
- **The name and dates add a little, but equally before and after:** +0.024 before and +0.026 after, both with ranges touching zero. Memory would show only before the cutoff, so this isn't memory.
- **Date only, before the cutoff: 0.525** (range 0.497 to 0.552), against 0.497 after. That's the direction memory would point, but the range includes 0.50, Gemma showed the opposite, and accuracy is no better than the fair bar (p = 0.91). Too weak to call memory.
- **The fake-data check came out low:** AUC 0.422, with a range (0.359 to 0.483) just below 0.50. A leak of future data would push AUC *up*, not down, so this doesn't point to a leak. With several checks run, one landing just outside its range by chance is expected now and then. It's recorded here, not explained away.
- **Reading the answers:** 983 replies were "Up." with a full stop. The first version of this notebook didn't strip the full stop and dropped them. It now does, and every answer is used. P(up) was never affected.

**Gemma and Llama 2 together:** neither model predicts the next day from prices, and neither shows convincing memory. Both answer "up" almost every time, so their accuracy just repeats the fair bar.